# Imports & Functions

In [1]:
import pandas as pd
import glob
import os

In [2]:
def create_id(df, columns, new_column_name, separator=""):
    df[new_column_name] = df[columns].astype(str).agg(separator.join, axis=1)
    return df

In [35]:
def abstract_result_columns(results, side):
    assert side in ["W", "L"], "side must be 'W' or 'L'"
    assert isinstance(results, pd.DataFrame), "results must be a DataFrame"
    
    df = results.copy()

    team_chr = "team_"
    opp_chr = "opp_"
    
    w_chr = team_chr if side == "W" else opp_chr
    l_chr = opp_chr if side == "W" else team_chr
    
    # Rename W columns (excluding WLoc)
    df.columns = [
        w_chr + col[1:] if col.startswith("W") and col != "WLoc" else col
        for col in df.columns
    ]
    
    # Rename L columns
    df.columns = [
        l_chr + col[1:] if col.startswith("L") else col
        for col in df.columns
    ]
    
    df["team_won"] = 1 if side == "W" else 0
    
    return df

# Load Data

In [3]:
# Path to the folder
folder_path = os.path.join(os.getcwd(), "../data/raw/march-machine-learning-mania-2026")

# Find all CSV files starting with "M"
pattern = os.path.join(folder_path, "M*.csv")
csv_files = glob.glob(pattern)

In [26]:
if not csv_files:
    print("No CSV files starting with 'M' found in the specified folder.")
else:
    dataframes = {}
    for file in csv_files:
        filename = os.path.basename(file)
        df = pd.read_csv(file)
        dataframes[filename] = df
        print(f"Loaded '{filename}': {df.shape[0]} rows, {df.shape[1]} columns")

Loaded 'MConferenceTourneyGames.csv': 6793 rows, 5 columns
Loaded 'MGameCities.csv': 90684 rows, 6 columns
Loaded 'MMasseyOrdinals.csv': 5761702 rows, 5 columns
Loaded 'MNCAATourneyCompactResults.csv': 2585 rows, 8 columns
Loaded 'MNCAATourneyDetailedResults.csv': 1449 rows, 34 columns
Loaded 'MNCAATourneySeedRoundSlots.csv': 776 rows, 5 columns
Loaded 'MNCAATourneySeeds.csv': 2626 rows, 3 columns
Loaded 'MNCAATourneySlots.csv': 2586 rows, 4 columns
Loaded 'MRegularSeasonCompactResults.csv': 196823 rows, 8 columns
Loaded 'MRegularSeasonDetailedResults.csv': 122775 rows, 34 columns
Loaded 'MSeasons.csv': 42 rows, 6 columns
Loaded 'MSecondaryTourneyCompactResults.csv': 1865 rows, 9 columns
Loaded 'MSecondaryTourneyTeams.csv': 1895 rows, 3 columns
Loaded 'MTeamCoaches.csv': 13898 rows, 5 columns
Loaded 'MTeamConferences.csv': 13753 rows, 3 columns
Loaded 'MTeams.csv': 381 rows, 4 columns
Loaded 'MTeamSpellings.csv': 1178 rows, 2 columns


In [27]:
df_reg_detail_results = dataframes.get('MRegularSeasonDetailedResults.csv').copy()
df_reg_compact_results = dataframes.get('MRegularSeasonCompactResults.csv').copy()

In [28]:
df_reg_detail_results = create_id(df_reg_detail_results, ['Season', 'WTeamID', 'LTeamID'], 'id', separator="_")
df_reg_compact_results = create_id(df_reg_compact_results, ['Season', 'WTeamID', 'LTeamID'], 'id', separator="_")

# Base summary stats

In [29]:
df_wins = df_reg_compact_results[['Season', 'WTeamID', 'WScore']].assign(wins=1).set_axis(['season', 'team_id', 'points_in_wins', 'wins'], axis=1)
df_wins_agg = df_wins.groupby(['season', 'team_id']).sum().reset_index()

df_losses = df_reg_compact_results[['Season', 'LTeamID', 'LScore']].assign(losses=1).set_axis(['season', 'team_id', 'points_in_losses', 'losses'], axis=1)
df_losses_agg = df_losses.groupby(['season', 'team_id']).sum().reset_index()

df_results_agg = df_wins_agg.merge(df_losses_agg, on = ['season', 'team_id'], how = 'outer')
df_results_agg['points'] = df_results_agg['points_in_wins'] + df_results_agg['points_in_losses']
df_results_agg = df_results_agg.fillna(0)


In [39]:
df_reg_team_results = pd.concat([
    abstract_result_columns(df_reg_detail_results, side = 'W'),
    abstract_result_columns(df_reg_detail_results, side = 'L')
])

In [40]:
df_reg_team_results

,Season,DayNum,team_TeamID,team_Score,opp_TeamID,opp_Score,WLoc,NumOT,team_FGM,team_FGA,...,opp_FTA,opp_OR,opp_DR,opp_Ast,opp_TO,opp_Stl,opp_Blk,opp_PF,id,team_won
0,2003,10,1104,68,1328,62,N,0,27,58,...,22,10,22,8,18,9,2,20,2003_1104_1328,1
1,2003,10,1272,70,1393,63,N,0,26,62,...,20,20,25,7,12,8,6,16,2003_1272_1393,1
2,2003,11,1266,73,1437,61,N,0,24,58,...,23,31,22,9,12,2,5,23,2003_1266_1437,1
3,2003,11,1296,56,1457,50,N,0,18,38,...,15,17,20,9,19,4,3,23,2003_1296_1457,1
4,2003,11,1400,77,1208,71,N,0,30,61,...,27,21,15,12,10,7,1,14,2003_1400_1208,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122770,2026,93,1347,78,1457,80,A,0,28,66,...,21,8,23,18,11,9,1,17,2026_1457_1347,0
122771,2026,93,1440,67,1459,81,A,0,23,58,...,24,7,30,15,8,3,3,19,2026_1459_1440,0
122772,2026,93,1236,61,1464,90,A,0,24,52,...,6,4,23,16,11,4,2,15,2026_1464_1236,0
122773,2026,93,1355,62,1472,77,A,0,23,52,...,12,0,24,10,4,5,0,13,2026_1472_1355,0
